# Web Application Security Scanner

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import requests
import re
import urllib.parse
import threading
import json
import time
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser
import ssl
import socket
from datetime import datetime

class WebSecurityScanner:
    def __init__(self, target_url):
        self.target_url = target_url.rstrip('/')
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })
        self.vulnerabilities = []
        self.discovered_urls = set()
        self.forms = []
        self.cookies = []
        self.headers_analysis = {}

    def crawl_website(self, max_depth=2):
        self.crawl_recursive(self.target_url, 0, max_depth)

    def crawl_recursive(self, url, depth, max_depth):
        if depth > max_depth or url in self.discovered_urls:
            return

        try:
            response = self.session.get(url, timeout=10, allow_redirects=True)
            self.discovered_urls.add(url)

            if response.headers.get('content-type', '').startswith('text/html'):
                soup = BeautifulSoup(response.text, 'html.parser')

                for link in soup.find_all('a', href=True):
                    next_url = urllib.parse.urljoin(url, link['href'])
                    if self.is_same_domain(next_url):
                        self.crawl_recursive(next_url, depth + 1, max_depth)

                self.extract_forms(soup, url)

        except Exception as e:
            pass

    def is_same_domain(self, url):
        target_domain = urllib.parse.urlparse(self.target_url).netloc
        url_domain = urllib.parse.urlparse(url).netloc
        return target_domain == url_domain

    def extract_forms(self, soup, url):
        forms = soup.find_all('form')
        for form in forms:
            form_data = {
                'url': url,
                'method': form.get('method', 'get').lower(),
                'action': urllib.parse.urljoin(url, form.get('action', '')),
                'inputs': []
            }

            for input_tag in form.find_all(['input', 'textarea', 'select']):
                input_data = {
                    'name': input_tag.get('name'),
                    'type': input_tag.get('type', 'text'),
                    'value': input_tag.get('value', '')
                }
                form_data['inputs'].append(input_data)

            self.forms.append(form_data)

    def test_sql_injection(self):
        payloads = [
            "' OR '1'='1", "' OR 1=1--", "' UNION SELECT NULL--",
            "'; DROP TABLE users--", "' OR 'a'='a", "1' OR '1'='1",
            "admin'--", "' OR 1=1#", "1' AND 1=1--"
        ]

        for form in self.forms:
            for payload in payloads:
                try:
                    data = {}
                    for input_field in form['inputs']:
                        if input_field['type'] in ['text', 'email', 'password']:
                            data[input_field['name']] = payload
                        else:
                            data[input_field['name']] = input_field['value']

                    if form['method'] == 'post':
                        response = self.session.post(form['action'], data=data, timeout=10)
                    else:
                        response = self.session.get(form['action'], params=data, timeout=10)

                    error_patterns = [
                        'sql syntax', 'mysql_fetch', 'ora-', 'postgresql',
                        'sqlite_', 'microsoft jet database', 'odbc error',
                        'syntax error', 'quoted string not properly terminated'
                    ]

                    for pattern in error_patterns:
                        if pattern in response.text.lower():
                            self.vulnerabilities.append({
                                'type': 'SQL Injection',
                                'url': form['action'],
                                'payload': payload,
                                'severity': 'High',
                                'description': f'SQL error detected with payload: {payload}'
                            })
                            break

                except Exception as e:
                    continue

    def test_xss(self):
        payloads = [
            '<script>alert("XSS")</script>',
            '<img src=x onerror=alert("XSS")>',
            '<svg onload=alert("XSS")>',
            'javascript:alert("XSS")',
            '<body onload=alert("XSS")>',
            '<iframe src="javascript:alert(\'XSS\')">',
            '\"><script>alert("XSS")</script>'
        ]

        for form in self.forms:
            for payload in payloads:
                try:
                    data = {}
                    for input_field in form['inputs']:
                        if input_field['type'] in ['text', 'email', 'search']:
                            data[input_field['name']] = payload
                        else:
                            data[input_field['name']] = input_field['value']

                    if form['method'] == 'post':
                        response = self.session.post(form['action'], data=data, timeout=10)
                    else:
                        response = self.session.get(form['action'], params=data, timeout=10)

                    if payload in response.text:
                        self.vulnerabilities.append({
                            'type': 'Cross-Site Scripting (XSS)',
                            'url': form['action'],
                            'payload': payload,
                            'severity': 'Medium',
                            'description': f'XSS payload reflected: {payload}'
                        })

                except Exception as e:
                    continue

    def test_directory_traversal(self):
        payloads = [
            '../../../etc/passwd',
            '..\\..\\..\\windows\\system32\\drivers\\etc\\hosts',
            '../../../boot.ini',
            '../../../../etc/shadow',
            '../../../proc/version'
        ]

        for url in list(self.discovered_urls)[:10]:
            for payload in payloads:
                try:
                    test_url = f"{url}?file={payload}"
                    response = self.session.get(test_url, timeout=10)

                    indicators = ['root:', 'daemon:', '[boot loader]', 'Linux version']

                    for indicator in indicators:
                        if indicator in response.text:
                            self.vulnerabilities.append({
                                'type': 'Directory Traversal',
                                'url': test_url,
                                'payload': payload,
                                'severity': 'High',
                                'description': f'Directory traversal successful with: {payload}'
                            })
                            break

                except Exception as e:
                    continue

    def analyze_security_headers(self):
        try:
            response = self.session.get(self.target_url, timeout=10)
            headers = response.headers

            security_headers = {
                'X-Frame-Options': 'Clickjacking protection',
                'X-XSS-Protection': 'XSS protection',
                'X-Content-Type-Options': 'MIME type sniffing protection',
                'Strict-Transport-Security': 'HTTPS enforcement',
                'Content-Security-Policy': 'XSS and injection protection',
                'Referrer-Policy': 'Referrer information control',
                'Permissions-Policy': 'Feature policy control'
            }

            self.headers_analysis = {
                'present': {},
                'missing': {}
            }

            for header, description in security_headers.items():
                if header in headers:
                    self.headers_analysis['present'][header] = {
                        'value': headers[header],
                        'description': description
                    }
                else:
                    self.headers_analysis['missing'][header] = description
                    self.vulnerabilities.append({
                        'type': 'Missing Security Header',
                        'url': self.target_url,
                        'header': header,
                        'severity': 'Low',
                        'description': f'Missing {header} header: {description}'
                    })

        except Exception as e:
            pass

    def check_ssl_certificate(self):
        try:
            parsed_url = urllib.parse.urlparse(self.target_url)
            if parsed_url.scheme == 'https':
                hostname = parsed_url.hostname
                port = parsed_url.port or 443

                context = ssl.create_default_context()
                with socket.create_connection((hostname, port), timeout=10) as sock:
                    with context.wrap_socket(sock, server_hostname=hostname) as ssock:
                        cert = ssock.getpeercert()

                        not_after = datetime.strptime(cert['notAfter'], '%b %d %H:%M:%S %Y %Z')
                        days_until_expiry = (not_after - datetime.now()).days

                        if days_until_expiry < 30:
                            self.vulnerabilities.append({
                                'type': 'SSL Certificate',
                                'url': self.target_url,
                                'severity': 'Medium',
                                'description': f'Certificate expires in {days_until_expiry} days'
                            })

        except Exception as e:
            self.vulnerabilities.append({
                'type': 'SSL Certificate',
                'url': self.target_url,
                'severity': 'High',
                'description': 'SSL certificate validation failed'
            })

    def test_http_methods(self):
        methods = ['OPTIONS', 'PUT', 'DELETE', 'TRACE', 'CONNECT', 'HEAD']

        for method in methods:
            try:
                response = self.session.request(method, self.target_url, timeout=10)
                if response.status_code < 400:
                    if method in ['PUT', 'DELETE', 'TRACE']:
                        self.vulnerabilities.append({
                            'type': 'Dangerous HTTP Method',
                            'url': self.target_url,
                            'method': method,
                            'severity': 'Medium',
                            'description': f'{method} method is allowed'
                        })

            except Exception as e:
                continue

    def check_robots_txt(self):
        try:
            robots_url = f"{self.target_url}/robots.txt"
            response = self.session.get(robots_url, timeout=10)

            if response.status_code == 200:
                disallowed_paths = []
                for line in response.text.split('\n'):
                    if line.strip().startswith('Disallow:'):
                        path = line.strip().split(':', 1)[1].strip()
                        if path and path != '/':
                            disallowed_paths.append(path)

                if disallowed_paths:
                    self.vulnerabilities.append({
                        'type': 'Information Disclosure',
                        'url': robots_url,
                        'severity': 'Low',
                        'description': f'Robots.txt reveals paths: {", ".join(disallowed_paths[:5])}'
                    })

        except Exception as e:
            pass

    def scan(self):
        print(f"Starting security scan of {self.target_url}")

        print("Crawling website...")
        self.crawl_website()
        print(f"Discovered {len(self.discovered_urls)} URLs")

        print("Testing for SQL injection...")
        self.test_sql_injection()

        print("Testing for XSS...")
        self.test_xss()

        print("Testing for directory traversal...")
        self.test_directory_traversal()

        print("Analyzing security headers...")
        self.analyze_security_headers()

        print("Checking SSL certificate...")
        self.check_ssl_certificate()

        print("Testing HTTP methods...")
        self.test_http_methods()

        print("Checking robots.txt...")
        self.check_robots_txt()

        self.generate_report()

    def generate_report(self):
        report = {
            'target': self.target_url,
            'scan_time': datetime.now().isoformat(),
            'urls_discovered': len(self.discovered_urls),
            'forms_found': len(self.forms),
            'total_vulnerabilities': len(self.vulnerabilities),
            'vulnerability_summary': self.get_vulnerability_summary(),
            'security_headers': self.headers_analysis,
            'vulnerabilities': self.vulnerabilities
        }

        with open('web_security_report.json', 'w') as f:
            json.dump(report, f, indent=2)

        print(f"\nScan completed!")
        print(f"Total vulnerabilities found: {len(self.vulnerabilities)}")
        print(f"High severity: {len([v for v in self.vulnerabilities if v.get('severity') == 'High'])}")
        print(f"Medium severity: {len([v for v in self.vulnerabilities if v.get('severity') == 'Medium'])}")
        print(f"Low severity: {len([v for v in self.vulnerabilities if v.get('severity') == 'Low'])}")
        print("Report saved to web_security_report.json")

    def get_vulnerability_summary(self):
        summary = {}
        for vuln in self.vulnerabilities:
            vuln_type = vuln['type']
            if vuln_type not in summary:
                summary[vuln_type] = 0
            summary[vuln_type] += 1
        return summary

def main():
    target_url = input("Enter target URL (e.g., https://example.com): ")

    if not target_url.startswith(('http://', 'https://')):
        target_url = 'https://' + target_url

    scanner = WebSecurityScanner(target_url)
    scanner.scan()

if __name__ == "__main__":
    main()